# Day 79 — Pytorch Datasets

This notebook covers PyTorch Dataset Abstraction (`torch.utils.data.Dataset` and `TensorDataset`), Custom Dataset Implementation (`__init__`, `__len__`, `__getitem__`), Tensor Precision Hygiene (`torch.float32`), and Regression Target Shape Alignment (`.reshape(-1, 1)`)

## Validation goals
- Understand why decoupling data indexing (`Dataset`) from batching and iteration (`DataLoader`) improves modularity, reusability, and pipeline maintainability
- Construct in-memory datasets using `torch.utils.data.TensorDataset` for aligned feature and target tensors
- Subclass `torch.utils.data.Dataset` (`TabularDemandDataset`) by implementing `__init__`, `__len__`, and `__getitem__` for custom tabular feature matrices and regression targets
- Enforce explicit `torch.float32` casting and 2D target reshaping (`.reshape(-1, 1)`) to prevent dtype and silent broadcasting bugs

In real ML workflows, however, keeping raw feature arrays, target labels, file paths, or transformation logic tangled together inside a training loop gets messy fast. Today we cover PyTorch Datasets (`torch.utils.data.Dataset` and `TensorDataset`), which decouple data representation and indexing from training loop logic.

### 1. Why Use a Dataset Abstraction?

A PyTorch Dataset acts as a uniform interface over your data. Whether your data comes from in-memory tensors, CSV files on disk, images, or custom feature pipelines, any Dataset exposes two core operations:
- How many items exist? (__len__)
- How do I retrieve sample idx? (__getitem__)

Separating data access from the model training loop gives you three practical advantages:
- Clean Modularity: Data loading and feature/target formatting live in one place instead of cluttering `train()`.
- Reusability: The exact same Dataset class can back training, validation, and testing splits with different underlying data.
- Standard Integration: DataLoader expects a Dataset object so it can handle batching, shuffling, and parallel fetching automatically.

### 2. Quick Start: TensorDataset for In-Memory Tensors

When your features and targets are already numerical tensors of the same first dimension (`N samples`), torch.utils.data.TensorDataset gives you a ready-made dataset without writing boilerplate.

In [1]:
import torch
from torch.utils.data import TensorDataset

# Suppose we have 5 retail samples with 4 numeric features and 1 target column
X = torch.tensor([
    [1.2, -0.4, 0.8, 120.0],
    [-0.5, 0.7, -0.2, 85.0],
    [0.9, 0.1, 1.4, 210.0],
    [-1.1, -0.8, 0.3, 60.0],
    [0.4, 0.5, -0.6, 145.0]
], dtype=torch.float32)

# Target shape (N, 1) for regression tasks
y = torch.tensor([[130.0], [78.0], [225.0], [55.0], [150.0]], dtype=torch.float32)

dataset = TensorDataset(X, y)

sample_x, sample_y = dataset[0]
print("--- Day 79: TensorDataset ---")
print(f"Dataset length: {len(dataset)}")
print(f"Sample 0 X:     {sample_x} (shape: {sample_x.shape})")
print(f"Sample 0 y:     {sample_y} (shape: {sample_y.shape})")

--- Day 79: TensorDataset ---
Dataset length: 5
Sample 0 X:     tensor([  1.2000,  -0.4000,   0.8000, 120.0000]) (shape: torch.Size([4]))
Sample 0 y:     tensor([130.]) (shape: torch.Size([1]))


### 3. Writing a Custom Dataset Class

For real-world datasets—such as loading preprocessed CSV splits like retail_demand_train_processed.csv or applying custom tensor conversions—subclassing `torch.utils.data.Dataset` gives you full control.

Every custom Dataset must implement three methods:

- __init__(self, ...): Stores the underlying data, paths, or transforms, and converts arrays into properly typed tensors (float32 for features and regression targets).
- __len__(self): Returns the total number of samples (int).
- __getitem__(self, idx): Returns a single sample at index idx, typically as a (features, target) tuple.

In [2]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset


class TabularDemandDataset(Dataset):
    """Custom PyTorch Dataset for tabular feature matrices and continuous targets."""

    def __init__(self, features: np.ndarray, targets: np.ndarray):
        # Ensure contiguous float32 tensors for PyTorch layers and loss functions
        self.X = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(targets, dtype=torch.float32).reshape(-1, 1)

    def __len__(self) -> int:
        return self.X.shape[0]

    def __getitem__(self, idx: int):
        return self.X[idx], self.y[idx]


# Example using synthetic data matching our preprocessed retail feature structure
rng = np.random.default_rng(42)
X_dummy = rng.normal(size=(100, 12))
y_dummy = rng.uniform(20.0, 400.0, size=(100,))

custom_ds = TabularDemandDataset(X_dummy, y_dummy)
x_item, y_item = custom_ds[2]

print("--- Day 79: Custom Dataset ---")
print(f"Total rows: {len(custom_ds)}")
print(f"Item 2 X shape: {x_item.shape}, dtype: {x_item.dtype}")
print(f"Item 2 y shape: {y_item.shape}, value: {y_item.item():.2f}")

--- Day 79: Custom Dataset ---
Total rows: 100
Item 2 X shape: torch.Size([12]), dtype: torch.float32
Item 2 y shape: torch.Size([1]), value: 86.93


If you load from a CSV file on disk (for `example pd.read_csv("retail_demand_train_processed.csv")`), you separate the target column (`demand_units`) from the feature columns first, then pass the resulting NumPy arrays into TabularDemandDataset.

### 4. Common Pitfalls to Watch For

Target Shape Mismatch: Regression loss functions like `nn.MSELoss()` often expect predictions and targets to share the same shape—either both (`batch_size, 1`) or both (`batch_size`,). Reshaping targets to (`-1, 1`) avoids silent broadcasting bugs when a model outputs shape (`batch_size, 1`).
- Wrong dtype: Passing float64 (NumPy's default float type) into a float32 linear layer causes a dtype mismatch error in PyTorch. Always cast explicitly to torch.float32.
- Mutating Data Inside __getitem__: Keep __getitem__ fast and deterministic. Heavy file parsing or full-dataset scaling belongs in preprocessing or __init__, not per-sample indexing.

### Guided Question:

Why is separating data indexing (`Dataset`) from batching and iteration (`DataLoader`) cleaner than writing raw array slicing inside your training loop?

### Answer:

Separating what one sample looks like (the `Dataset`) from how samples are grouped, shuffled, and fed (the `DataLoader`) keeps each piece focused on a single job instead of tangling data logic inside your model’s training loop.

**1. Single Responsibility & Readability**

Your training loop should read like high-level orchestration: grab a batch, run forward/backward passes, update weights. When raw slicing (X[i:i+batch_size], y[i:i+batch_size]) or file loading lives directly inside the loop, every change—like normalizing inputs, augmenting images, or reading from disk instead of RAM—forces you to edit loop logic and risk bugs.
- `Dataset` answers: "Given index i, give me one item (and its label/target)."
- `DataLoader` answers: "How should we shuffle indices, group them into batches, and iterate over them?"

**2. Shuffling, Batching, and Edge Cases Handled Automatically**
- Writing manual slicing often introduces subtle mistakes:
Shuffling: Shuffling X and y in place can desync features from labels if you aren't careful, or mutate your original dataset across epochs. A sampler/loader handles index shuffling cleanly each epoch.
- Uneven batch sizes: If you have 1,003 samples and a batch size of 32, the last batch only has 11 items. DataLoader handles partial final batches (or drops them via drop_last) consistently without off-by-one errors.
- Collation: Combining individual items (like variable-length sequences or dictionaries) into a single batch tensor is centralized rather than ad-hoc.

**3. Reusability Across Train, Validation, and Test**

With a Dataset, the exact same class can wrap your training split, validation split, and test split—just initialized with different data or transforms (e.g., data augmentation enabled for training, turned off for validation). You don't have to duplicate slicing code for every evaluation loop.

**4. Performance & Scaling (Prefetching / Workers)**

If your data isn't a single small NumPy array in memory—for instance, thousands of images or audio files on disk—slicing in the main training loop makes the GPU sit idle while the CPU reads and decodes files. A DataLoader can prefetch batches in background worker processes and pin memory for faster host-to-device transfers without cluttering your training code.


### Guided Question:

What are the exact roles of __len__ and __getitem__ in a custom `torch.utils.data`.Dataset, and what happens if they disagree on dataset size?

### Answer:

In a custom torch.utils.data.Dataset, __len__ and __getitem__ work as a strict pair: __len__ defines the size of the dataset, and __getitem__ defines how to fetch a single sample at a given index.

**1. Exact Role of __len__(self)**

__len__ returns a non-negative integer (`int`) representing the total number of samples available in the dataset.
- Who calls it: Samplers and `DataLoader` call l`en(dataset)` (which invokes __len__) to know the valid index range `[0, len(dataset) - 1]` and to determine how many batches make up one epoch.
- Best practice: Return a plain Python int (for example, `len(self.samples)` `or self.inputs.shape[0]`).

**2. Exact Role of __getitem__(self, idx)**

__getitem__ takes an index idx and returns one sample (commonly a tuple (input_data, target) or a dictionary), usually after loading/parsing, applying transforms, and converting to tensors.
- Who calls it: DataLoader (and its worker processes) repeatedly calls dataset[idx] for indices chosen by a sampler, then stacks/collates those individual items into a batch.
- Important nuance (idx type): Standard samplers pass an integer idx, though direct indexing on a dataset can also pass a slice or list/array of indices if you explicitly handle it. Usually, writing __getitem__ around a single integer index idx is cleanest and most reliable.

### Guided Question:

Why do we explicitly specify `dtype=torch.float32` when converting NumPy arrays to tensors for neural network training, and why is `.reshape(-1, 1)` helpful for regression targets?

### Answer:

**1. Why Specify `dtype=torch.float32`?**

NumPy defaults to float64 (double precision) for floating-point arrays, whereas PyTorch neural networks default to float32 (single precision). Explicitly setting `dtype=torch.float32` avoids silent mismatches and wasted compute:
- Parameter & layer compatibility: Model weights and biases in PyTorch default to float32. Passing a float64 tensor into a float32 layer (like `nn.Linear`) triggers a RuntimeError: mat1 and mat2 must have the same dtype (or similar type error).
- Speed & GPU efficiency: float32 uses half the memory of float64, speeds up matrix math dramatically, and runs natively on GPUs—many consumer and cloud GPUs have far fewer float64 compute units than float32.
- Controlling precision intentionally: Explicit `dtype=torch.float32` makes your data pipeline predictable instead of depending on whatever dtype NumPy happened to infer (for example, np.array([1, 2, 3]) has integer dtype int64, not float).

**2. Why `.reshape(-1, 1)` Is Helpful for Regression Targets**

A 1D target array of shape (N,) (like `y.shape == (100,)`) often causes shape mismatches when compared against model predictions from layers like nn.Linear(in_features, 1), which output shape (`batch_size`, 1).

What `.reshape(-1, 1)` does: -1 tells PyTorch/NumPy to infer the sample count automatically (N), while 1 makes the second dimension explicit—turning (N,) into (N, 1) (a column vector of N rows and 1 target per row).

**Why that prevents bugs:**

- Loss functions: Losses like nn.MSELoss() or nn.L1Loss() can silently broadcast (batch_size, 1) against (batch_size,)—sometimes masking shape bugs—or fail when shapes don't align cleanly. Matching shapes (batch_size, 1) explicitly keeps expectations clear.
- Consistency with multi-output models: If you later predict 2 or 3 targets per sample ((batch_size, 3)), keeping a 2D (batch_size, target_dim) shape for a single regression target ((batch_size, 1)) makes your pipeline consistent.

**Quick Example**

In [3]:
import numpy as np
import torch

X_np = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])  # float64, shape (3, 2)
y_np = np.array([10.0, 20.0, 30.0])                  # float64, shape (3,)

X = torch.tensor(X_np, dtype=torch.float32)              # float32, shape (3, 2)
y = torch.tensor(y_np, dtype=torch.float32).reshape(-1, 1)  # float32, shape (3, 1)